# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and the [Croissant schema](https://mlcommons.org/croissant/).

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n\nPublished: {metadata.datePublished}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets and their IDs, fields, and columns. We use `@id` fields for referencing.

In [ ]:
# List all record sets with their @id and field @ids
record_sets = metadata.record_sets
print(f"Total record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    # List columns and fields
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Field @ids: {[field['@id'] for field in fields]}")
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    print(f"  Column @ids: {[column['@id'] for column in columns]}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Prepare to extract data for all record sets
import pprint

record_set_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set: {record_set_id}, shape: {dataframes[record_set_id].shape}")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head(), '\n')
    except Exception as e:
        print(f"Could not load records from {record_set_id}: {e}\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

> **Note:** For this example, we will analyze the first (main) record set if available.

In [ ]:
### Choose a record set for analysis
if len(record_set_ids) == 0:
    raise RuntimeError('No record sets found in the schema.')
record_set_id = record_set_ids[0]
df = dataframes.get(record_set_id)
if df is None or df.empty:
    raise RuntimeError(f'No records found in record set {record_set_id}.')
print(f"Analyzing record set: {record_set_id}")

# List columns to help choose a numeric field
print("Columns in DataFrame:", df.columns.tolist())
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
if not numeric_columns:
    # Attempt to infer numeric columns
    possible_numeric = []
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            possible_numeric.append(col)
        except Exception:
            pass
    numeric_columns = possible_numeric

print('Numeric columns detected:', numeric_columns)

# Choose first numeric column for demonstration
if numeric_columns:
    numeric_field = numeric_columns[0]
else:
    raise RuntimeError('No numeric fields detected for analysis.')

threshold = df[numeric_field].mean() if not pd.isnull(df[numeric_field].mean()) else 0
# Filter records above threshold (common for expression/abundance)
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} records")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field}_zscore"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records (z-score):")
print(filtered_df[[numeric_field, f"{numeric_field}_zscore"]].head())

# Group by a categorical field if it exists (e.g. sample_id, protein_type, etc.)
group_field = None
categorical_candidates = [c for c in df.columns if df[c].dtype == object]
if categorical_candidates:
    group_field = categorical_candidates[0]
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped mean of {numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the chosen numeric field and its normalized variant, and visualize grouping if relevant.

You can adjust the visualization for columns most relevant to your analysis.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field} (Filtered > Mean)")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# If grouping was done, make a bar plot of top groups
if group_field:
    top_groups = grouped_df.sort_values(ascending=False).head(10)
    plt.figure(figsize=(8, 5))
    sns.barplot(x=top_groups.values, y=top_groups.index)
    plt.title(f"Mean {numeric_field} by {group_field} (Top 10 Groups)")
    plt.xlabel(numeric_field)
    plt.ylabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and visualize the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library and the Croissant schema. By leveraging the `@id` fields, we dynamically discovered and referenced record sets and their fields, performed basic data cleaning steps, normalization, grouping, and generated quick visualizations for domain analysis.

Data schemas published with Croissant standard make it easy to automate exploration and reproducible pipelines for FAIR datasets.